# ATML PA2 - Task 4 (safety calibration) GPU stage on Kaggle (2x T4)

This notebook only **orchestrates** the repository scripts (`task4_safety/`). It runs the GPU work only:
deterministic XSTest responses (+ reward scores) for SFT / DPO / PPO / GRPO and the fixed AI judge
(Qwen2.5-3B-Instruct, 4-bit). The **manual audit and final evaluation run locally on CPU** afterwards.

**Before running:** Accelerator = *GPU T4 x2*, Internet = *On*, and **attach the Task 1-3 adapters as input**
(Add Input -> your dataset `atml-pa2-task4-adapters`, or the three `task*_artifacts.zip` files).
`task4_safety/run_all.sh` finds `outputs/task{1_dpo,2_ppo,3_grpo}/standard` anywhere under `/kaggle/input`.

| GPU 0 | GPU 1 |
|---|---|
| SFT + DPO: generate (greedy, 256 tokens) + reward score + judge | PPO + GRPO: same |

Output: `task4_artifacts.zip` (results/task4_safety + report/task4).

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/alihasnaink/atml-pa2.git"
BRANCH = "main"
WORKDIR = "/tmp/repo"   # outside /kaggle/working: keeps checkpoints out of the notebook output

import os, subprocess
url = REPO_URL
try:  # optional: private repository
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, WORKDIR], check=True)
else:  # an existing clone (e.g. from an earlier run) is synced to the latest pushed commit
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORKDIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 2. Install the pinned environment

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.check_environment
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 3. Download and validate the fixed course assets

In [ ]:
!python -m scripts.download_assets
!python -m scripts.validate_assets

## 4. Run the Task 4 GPU stage
Resumable within a session: re-running this cell skips finished runs/evaluations. Per-GPU logs: `results/task4_safety/logs/`.

In [ ]:
!bash task4_safety/run_all.sh

## 5. Quick look at the results

In [ ]:
from IPython.display import Markdown, Image, display
for t in ["safety_calibration", "category_over_refusal", "category_unsafe_compliance", "reward_within_prompt"]:
    p = f"report/task4/tables/{t}.md"
    if os.path.exists(p):
        display(Markdown(open(p).read()))
for f in ["label_distribution", "category_heatmap"]:
    p = f"report/task4/figures/{f}.png"
    if os.path.exists(p):
        display(Image(p))

## 6. Package the artifacts
Unzip `task4_artifacts.zip` at the repository root locally, then run `python -m task4_safety.make_audit_sheet`, label the sheet by hand, and run `python -m task4_safety.evaluate_safety`.

In [ ]:
!cd /tmp/repo && zip -qr /kaggle/working/task4_artifacts.zip results/task4_safety report/task4
!ls -lh /kaggle/working/